In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langchain_google_genai import ChatGoogleGenerativeAI


In [3]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    temperature=0.5
)

In [4]:
from langchain_community.document_loaders import PyPDFLoader

pdf_path="./docs/Nvidia_Annual_Report_2026.pdf"
loader = PyPDFLoader(pdf_path)

docs =loader.load()
docs

[Document(metadata={'producer': 'Adobe Acrobat (64-bit) 26.1.21529', 'creator': 'Adobe Acrobat (64-bit) 26.1.21529', 'creationdate': '2026-05-14T08:02:35-04:00', 'moddate': '2026-05-14T08:03:02-04:00', 'source': './docs/Nvidia_Annual_Report_2026.pdf', 'total_pages': 175, 'page': 0, 'page_label': '1'}, page_content='2026\nNVIDIA Corporation\nAnnual Review\nNotice of Annual Meeting \nProxy Statement \nForm 10-K'),
 Document(metadata={'producer': 'Adobe Acrobat (64-bit) 26.1.21529', 'creator': 'Adobe Acrobat (64-bit) 26.1.21529', 'creationdate': '2026-05-14T08:02:35-04:00', 'moddate': '2026-05-14T08:03:02-04:00', 'source': './docs/Nvidia_Annual_Report_2026.pdf', 'total_pages': 175, 'page': 1, 'page_label': '2'}, page_content='For most of computing history, software was pre-recorded.\nHumans described an algorithm. Computers executed it.\nNow software is trained. It learns from data and generates  \nintelligence in real time.\nGeneral-purpose computing has run its course. Moore’s Law slowe

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=50
)

chunks = splitter.split_documents(docs)
chunks

[Document(metadata={'producer': 'Adobe Acrobat (64-bit) 26.1.21529', 'creator': 'Adobe Acrobat (64-bit) 26.1.21529', 'creationdate': '2026-05-14T08:02:35-04:00', 'moddate': '2026-05-14T08:03:02-04:00', 'source': './docs/Nvidia_Annual_Report_2026.pdf', 'total_pages': 175, 'page': 0, 'page_label': '1'}, page_content='2026\nNVIDIA Corporation\nAnnual Review\nNotice of Annual Meeting \nProxy Statement \nForm 10-K'),
 Document(metadata={'producer': 'Adobe Acrobat (64-bit) 26.1.21529', 'creator': 'Adobe Acrobat (64-bit) 26.1.21529', 'creationdate': '2026-05-14T08:02:35-04:00', 'moddate': '2026-05-14T08:03:02-04:00', 'source': './docs/Nvidia_Annual_Report_2026.pdf', 'total_pages': 175, 'page': 1, 'page_label': '2'}, page_content='For most of computing history, software was pre-recorded.\nHumans described an algorithm. Computers executed it.\nNow software is trained. It learns from data and generates  \nintelligence in real time.\nGeneral-purpose computing has run its course. Moore’s Law slowe

In [7]:
len(chunks)

753

In [8]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embedding_model = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")

In [ ]:
## Only run this when indexing for the first time 


from langchain_community.vectorstores import Chroma
import time

batch_size =20
vectorstore = None

for i in range(0,len(chunks),batch_size):
    batch = chunks[i:i+batch_size]

    retries =5

    while retries>0:
        try:
            if vectorstore is None:
                vectorstore =Chroma.from_documents(
                    documents=batch,
                    embedding=embedding_model,
                    persist_directory="./chroma_db"
                )
            else:
                vectorstore.add_documents(batch)
            
            print(f"✅ Indexed {min(i+batch_size,len(chunks))}/{len(chunks)}")
            break

        except Exception as e:
            retries-=1

            if retries==0:
                raise e
            print(f"⚠️ {e}")
            print("Retrying in 30 seconds...")
            time.sleep(30)
    time.sleep(2)
print("🎉 All chunks indexed successfully!")




✅ Indexed 20/753


KeyboardInterrupt: 

In [ ]:
#Instead of above indexing after once done comment above cell and uncomment this one, run this cell for any subsequent usage as embeddings will have been persisited the embeddings!
"""from langchain_community.vectorstores import Chroma

vectorstore = Chroma(
    persist_directory="./chroma_db",
    embedding_function=embedding_model
)"""

In [ ]:
question = "What kind of profit did Nvidia do in 2026 also give timeline"

context = vectorstore.similarity_search(question, k=3)

response = llm.invoke(
    f"""
    Answer the following question using only the provided context.

    Context:
    {context}

    Question:
    {question}
    """
)

print(response.content[0]['text'])

Based on the provided documents, Nvidia's financial results for the fiscal year 2026 are as follows:

*   **Gross Profit:** $153,463 million
*   **Operating Income:** $130,387 million (referenced as $130.4 billion in the Business Overview)
*   **Net Income:** $120,067 million

**Timeline:**
The fiscal year 2026 ended on **January 25, 2026**.
